# Loading Data

## Env

In [1]:
from pathlib import Path
import sys
import logging

project_root = str(Path("/home/user/perso/trading/alphalab").resolve())

if project_root not in sys.path:
    sys.path.append(project_root)

# Configuration de l'autocomplétion Jupyter
%config IPCompleter.use_jedi = False
%config IPCompleter.greedy = False

# Activation de l'autoreload
%load_ext autoreload
%autoreload 2

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)s] %(message)s',
    stream=sys.stdout,
    force=False,
)

logging.getLogger('enl').setLevel(logging.INFO)

## Load Line Profiler

In [ ]:
!pip install line_profiler

In [2]:
%load_ext line_profiler

# Optim

In [4]:
start = '2020-01-01'
end = '2020-12-31'

In [ ]:
from enl.data_feed import *

start = '2020-01-01'
end = '2020-12-31'

%timeit df_30min = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'])

## Line Profiler

In [ ]:
%lprun -f load_pair_time_bars df_30min = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'])


In [ ]:
%lprun -f load_time_bars df_30min = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'])


In [ ]:
%lprun -f load_ticks df_30min = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'])


# Optimizing load_ticks()

In [ ]:
%lprun -f load_ticks df_ticks = load_ticks('EURUSD', start, end, None)

## With thread optim

In [ ]:
from enl.data_feed import *

%time df_ticks = load_ticks('EURUSD', start, end, None)
%time df_ticks = load_ticks_optim('EURUSD', start, end, None)

## With loop optim

In [ ]:
from enl.data_feed import *

%time df_ticks = load_ticks('EURUSD', start, end, None)
%time df_ticks = load_ticks_myoptim('EURUSD', start, end, None)

In [ ]:
from enl.data_feed import *

%lprun -f load_ticks_myoptim df_ticks = load_ticks_myoptim('EURUSD', start, end, None)

## By Column

In [ ]:
for i in range(1, 5, 1):
    print(f'--- #{i} ---')
    %time df_ticks = load_ticks('EURUSD', start, end, columns=None)
    %time df_ticks = load_ticks('EURUSD', start, end, columns=['timestamp', 'price_mid'])


# Conclusion

Only load necessary columns!

# Optim load_time_bars

In [ ]:
from enl.data_feed import *

for i in range(1, 6, 1):
    print(f'--- #{i} ---')
    %time df = load_time_bars('EURUSD', start, end, '30min', ['open', 'high', 'low', 'close', 'volume_sum', 'spread_mean'])
    %time df2 = load_time_bars_optim('EURUSD', start, end, '30min', ['open', 'high', 'low', 'close', 'volume_sum', 'spread_mean'])
    # print(df.head(1))
    # print(df2.head(1))
    pd.testing.assert_frame_equal(df, df2)

In [ ]:
from enl.data_feed import *

for i in range(1, 6, 1):
    print(f'--- #{i} ---')
    %time df = load_time_bars('EURUSD', start, end, '30min', ['close'])
    %time df2 = load_time_bars_optim('EURUSD', start, end, '30min', ['close'])
    # print(df.head(1))
    # print(df2.head(1))
    pd.testing.assert_frame_equal(df, df2)

In [4]:
from enl.data_feed import *
start = '2020-01-01'
end = '2020-12-31'
%lprun -f load_time_bars df = load_time_bars('EURUSD', start, end, '30min', ['close'])

[WARNING] 5088 bars have been dropped


Timer unit: 1e-09 s

Total time: 5.74842 s
File: /home/user/perso/trading/alphalab/enl/data_feed.py
Function: load_time_bars at line 163

Line #      Hits         Time  Per Hit   % Time  Line Contents
   163                                           def load_time_bars(
   164                                               symbol: str,
   165                                               start_date: str,
   166                                               end_date: str,
   167                                               time_frame: str,
   168                                               columns: list[str] | None = None,
   169                                           ) -> pd.DataFrame:
   170                                               """Loads tick data and aggregates it into regular chronological time bars.
   171                                           
   172                                               Args:
   173                                                   symbol:

In [5]:
from enl.data_feed import *
start = '2020-01-01'
end = '2020-12-31'
%lprun -f load_time_bars_optim df = load_time_bars_optim('EURUSD', start, end, '30min', ['close'])

[WARNING] 5088 bars have been dropped


Timer unit: 1e-09 s

Total time: 3.89419 s
File: /home/user/perso/trading/alphalab/enl/data_feed.py
Function: load_time_bars_optim at line 246

Line #      Hits         Time  Per Hit   % Time  Line Contents
   246                                           def load_time_bars_optim(
   247                                               symbol: str,
   248                                               start_date: str,
   249                                               end_date: str,
   250                                               time_frame: str,
   251                                               columns: list[str] | None = None,
   252                                           ) -> pd.DataFrame:
   253                                               """Loads tick data and aggregates it into regular chronological time bars.
   254                                           
   255                                               Args:
   256                                              

## In load pair

In [7]:
from enl.data_feed import *

start = '2020-01-01'
end = '2020-12-31'

for i in range(1, 6, 1):
    print(f'--- #{i} ---')
    %time df_30min = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'], use_optim=False)
    %time df_30min = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'], use_optim=True)

--- #1 ---
[WARNING] 5090 bars have been dropped
[WARNING] 5088 bars have been dropped
[WARNING] Found 2 gaps in pair (DROP)
[WARNING] Dropped 2 unsync-able datetime
[INFO] Pair "GBPUSD (X) / EURUSD (Y)" loaded (12386 bars)
CPU times: user 13 s, sys: 3.13 s, total: 16.1 s
Wall time: 11 s
[WARNING] 5090 bars have been dropped
[WARNING] 5088 bars have been dropped
[WARNING] Found 2 gaps in pair (DROP)
[WARNING] Dropped 2 unsync-able datetime
[INFO] Pair "GBPUSD (X) / EURUSD (Y)" loaded (12386 bars)
CPU times: user 6.42 s, sys: 737 ms, total: 7.16 s
Wall time: 6.59 s
--- #2 ---
CPU times: user 1.41 s, sys: 279 ms, total: 1.69 s
Wall time: 848 ms


KeyboardInterrupt: 

In [7]:
from enl.data_feed import *

start = '2020-01-01'
end = '2024-12-31'

for i in range(1, 6, 1):
    print(f'--- #{i} ---')
    # %time df1 = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'], use_optim=True, use_thread=False)
    %time df2 = load_pair_time_bars('GBPUSD', 'EURUSD', start, end, '30min', ['close'], use_optim=True, use_thread=True)

--- #1 ---
[WARNING] File "/home/user/perso/trading/alphalab/data/EURUSD/raw/EURUSD_TICKS_2021_01_01.parquet" not found
[WARNING] File "/home/user/perso/trading/alphalab/data/EURUSD/raw/EURUSD_TICKS_2023_12_31.parquet" not found
[WARNING] File "/home/user/perso/trading/alphalab/data/GBPUSD/raw/GBPUSD_TICKS_2021_01_01.parquet" not found
[WARNING] File "/home/user/perso/trading/alphalab/data/GBPUSD/raw/GBPUSD_TICKS_2023_12_24.parquet" not found
[WARNING] File "/home/user/perso/trading/alphalab/data/GBPUSD/raw/GBPUSD_TICKS_2023_12_31.parquet" not found
[WARNING] 25578 bars have been dropped
[WARNING] 25586 bars have been dropped
[WARNING] Found 8 gaps in pair (DROP)
[WARNING] Dropped 8 unsync-able datetime
[INFO] Pair "GBPUSD (X) / EURUSD (Y)" loaded (62018 bars)
CPU times: user 31 s, sys: 5.1 s, total: 36.1 s
Wall time: 19.7 s
--- #2 ---
[WARNING] File "/home/user/perso/trading/alphalab/data/EURUSD/raw/EURUSD_TICKS_2021_01_01.parquet" not found
[WARNING] File "/home/user/perso/trading/al

KeyboardInterrupt: 